final summary


In [ ]:
summary = pd.DataFrame([
    {"Model": "Baseline CNN (from scratch)", "Accuracy": baseline_eval["acc"],
     "Macro Precision": baseline_eval["precision"], "Macro Recall": baseline_eval["recall"],
     "Macro F1": baseline_eval["f1"]},
    {"Model": "Transfer Learning (EfficientNetB0)", "Accuracy": transfer_eval["acc"],
     "Macro Precision": transfer_eval["precision"], "Macro Recall": transfer_eval["recall"],
     "Macro F1": transfer_eval["f1"]},
])
summary_display = summary.set_index("Model").round(4)
summary_display.to_csv(RESULTS_DIR / "final_summary.csv")
summary_display


In [ ]:
from google.colab import files as gfiles

def predict_image(model, classes, pil_image):
    # FIX: transfer_model (EfficientNetB0) expects raw [0,255] pixels — its internal
    # Rescaling/Normalization layer does the scaling. Do NOT divide by 255 here.
    img = pil_image.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    x = img_to_array(img)  # stays in [0,255]
    x = np.expand_dims(x, axis=0)
    probs = model.predict(x, verbose=0)[0]
    top_idx = probs.argsort()[-3:][::-1]
    return [(classes[i], float(probs[i])) for i in top_idx]

print("Upload an image of a vehicle to classify:")
uploaded_img = gfiles.upload()
img_path = list(uploaded_img.keys())[0]
pil_img = Image.open(img_path)

plt.figure(figsize=(4,4)); plt.imshow(pil_img); plt.axis("off"); plt.show()

results = predict_image(transfer_model, transfer_eval["classes"], pil_img)
print("\nTop-3 predictions:")
for label, conf in results:
    print(f"  {label:20s} {conf*100:.1f}%")
if results[0][1] < 0.5:
    print("\n[LOW CONFIDENCE] — flag for manual review rather than trusting automatically.")
